<a href="https://colab.research.google.com/github/861728/ReserveService/blob/main/naver_map_%EA%B2%80%EC%88%98%EC%9A%94%EC%9D%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 네이버 지도 전화번호 자동 수집기 v2
구글 계정 로그인만으로 스프레드시트에 접근합니다. (서비스 계정 불필요)

In [ ]:
# ✅ STEP 1: 패키지 설치
!pip install selenium gspread -q
!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!cp /usr/lib/chromium-browser/chromedriver /usr/bin/ 2>/dev/null || true

# 설치 확인
import subprocess
print(subprocess.run(['chromium-browser', '--version'], capture_output=True, text=True).stdout)
print(subprocess.run(['chromedriver', '--version'], capture_output=True, text=True).stdout)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:8 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,210 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,035 B in 1s (2,243 B/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Readin

In [ ]:
# ✅ STEP 2: 구글 계정 인증 (팝업 뜨면 본인 계정으로 로그인)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
client = gspread.authorize(creds)
print('✅ 인증 완료!')

✅ 인증 완료!


In [ ]:
# ✅ STEP 3: 스프레드시트 연결
# 본인 스프레드시트 URL 붙여넣기
SPREADSHEET_URL = 'https://docs.google.com/spreadsheets/d/1guEAEM8Vz0VfkTojJXneucjm1RJgsf0KjLTrE2tXFNY/edit?gid=0#gid=0'
SHEET_NAME = '26Y 대기예약 옵트아웃 타겟 검수'  # 탭 이름 (한글이면 그대로 입력, 예: '시트1')

sheet = client.open_by_url(SPREADSHEET_URL).worksheet(SHEET_NAME)
all_data = sheet.get_all_values()

print(f'✅ 연결 성공! 총 {len(all_data)-1}행 데이터')
print(f'헤더: {all_data[0]}')

✅ 연결 성공! 총 758행 데이터
헤더: ['SHOP_SEQ', 'SHOP_NAME', 'dining_yn', 'online_yn', 'N 연동', '주력서비스 1', '주력서비스 2', '예약 사용료', '대기 시간 구간', 'SI_DO', 'SI_GUN_GU', 'FOOD_KIND', '가맹일', '4분기(평균) 전체 등록 수', '4분기(평균) 원격 등록 수', '4분기(평균) 전체 완료 수', '4분기(평균) 원격 완료 수', '1월 예약 등록 수', '1월 예약 방문 수', '매장 티어', '상위 티어', '검수 요일', '검수 운영 시간', '브레이크타임', '담당자', '검수 일자', '', '', '']


In [ ]:
# ✅ STEP 4: 열 설정 확정
B_COL      = 2    # PLACE_NAME 열
J_COL      = 10   # 시 정보
K_COL      = 11   # 구 정보
CLOSED_COL = 22   # V열 - 휴무일
HOURS_COL  = 23   # W열 - 영업시간
BREAK_COL  = 24   # X열 - 브레이크타임

headers = all_data[0]
print(f'B열:  {headers[B_COL-1]}')
print(f'J열:  {headers[J_COL-1]}')
print(f'K열:  {headers[K_COL-1]}')
print(f'V열:  {headers[CLOSED_COL-1] if len(headers) >= CLOSED_COL else "(비어있음)"}')
print(f'W열:  {headers[HOURS_COL-1]  if len(headers) >= HOURS_COL  else "(비어있음)"}')
print(f'X열:  {headers[BREAK_COL-1]  if len(headers) >= BREAK_COL  else "(비어있음)"}')

B열:  SHOP_NAME
J열:  SI_DO
K열:  SI_GUN_GU
V열:  검수 요일
W열:  검수 운영 시간
X열:  브레이크타임


In [ ]:
# ✅ STEP 5: 셀레니움 드라이버 초기화

# 구글 크롬 직접 설치
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q
!google-chrome --version

# 크롬 버전 확인 후 맞는 chromedriver 설치
!google-chrome --version
!pip install webdriver-manager -q

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, re, urllib.parse

def create_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

driver = create_driver()
print('✅ 셀레니움 드라이버 초기화 완료!')

Reading package lists...
Building dependency tree...
Reading state information...
google-chrome-stable is already the newest version (146.0.7680.80-1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
Google Chrome 146.0.7680.80 
Google Chrome 146.0.7680.80 
✅ 셀레니움 드라이버 초기화 완료!


In [ ]:
# ✅ STEP 6: 셀레니움 드라이버 초기화
import re, time, urllib.parse, json
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def extract_json_array(src, key):
    pattern = f'"{key}"\\s*:\\s*\\['
    match = re.search(pattern, src)
    if not match:
        return None
    start = match.end() - 1
    depth = 0
    for i in range(start, len(src)):
        if src[i] == '[':
            depth += 1
        elif src[i] == ']':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(src[start:i+1])
                except:
                    return None
    return None

def get_place_id(src, driver):
    # 방법 1: 링크에서 직접 추출 (가장 정확)
    try:
        for sel in ['a[href*="/restaurant/"]', 'a[href*="/place/"]']:
            elems = driver.find_elements(By.CSS_SELECTOR, sel)
            for el in elems:
                href = el.get_attribute('href') or ''
                match = re.search(r'/(restaurant|place)/(\d+)', href)
                if match:
                    return match.group(2)
    except:
        pass

    # 방법 2: Restaurant:{id} 패턴
    match = re.search(r'Restaurant:(\d{6,})', src)
    if match:
        return match.group(1)

    # 방법 3: /restaurant/ URL 패턴
    match = re.search(r'/restaurant/(\d+)', src)
    if match:
        return match.group(1)

    # 방법 4: "id" JSON (마지막 수단)
    match = re.search(r'"id"\s*:\s*"(\d{6,})"', src)
    if match:
        return match.group(1)

    return None

def get_hours_from_naver(place_name, driver, gu=''):
    try:
        query = f'{place_name} {gu}'.strip() if gu else place_name
        encoded = urllib.parse.quote(query)

        # ── 1. place_id 추출 ──
        driver.get(f'https://pcmap.place.naver.com/place/list?query={encoded}')
        time.sleep(5)

        src = driver.page_source
        place_id = get_place_id(src, driver)

        if not place_id:
            print(f'[{place_name}] place_id 찾기 실패')
            return set(), '없음', '없음'

        print(f'[{place_name}] place_id: {place_id}')

        # ── 2. pcmap 상세 페이지 접근 ──
        driver.get(f'https://pcmap.place.naver.com/restaurant/{place_id}/home')
        time.sleep(5)

        # 404 체크 → /place/ 로 재시도
        if '찾을 수 없습니다' in driver.page_source:
            print(f'[{place_name}] /restaurant/ 404 → /place/ 재시도')
            driver.get(f'https://pcmap.place.naver.com/place/{place_id}/home')
            time.sleep(5)

        # ── 3. 영업시간 토글 버튼 클릭 ──
        toggle_clicked = False
        for sel in [
            'a[class*="OpenHour"]', 'a[class*="bizHour"]',
            'a[class*="moreBtn"]', 'span[class*="hours"] ~ a',
            'div[class*="hour"] a',
        ]:
            elems = driver.find_elements(By.CSS_SELECTOR, sel)
            if elems:
                driver.execute_script("arguments[0].click();", elems[0])
                time.sleep(1.5)
                toggle_clicked = True
                break

        if not toggle_clicked:
            for xp in [
                '//span[contains(text(),"영업시간")]/following::a[1]',
                '//span[contains(text(),"운영시간")]/following::a[1]',
                '//a[contains(@class,"time")]',
            ]:
                elems = driver.find_elements(By.XPATH, xp)
                if elems:
                    driver.execute_script("arguments[0].click();", elems[0])
                    time.sleep(1.5)
                    break

        # ── 4. 영업시간 + 휴무일 + 브레이크타임 파싱 ──
        src = driver.page_source
        time_values  = set()
        break_values = set()
        closed_days  = []

        raw = extract_json_array(src, 'businessHours')
        if raw:
            try:
                for item in raw:
                    bh    = item.get('businessHours') or {}
                    start = bh.get('start', '')
                    end   = bh.get('end', '')
                    desc  = item.get('description') or ''
                    day   = item.get('day', '')

                    if start and end:
                        time_values.add(f'{start}-{end}')
                        for brk in item.get('breakHours') or []:
                            bs = brk.get('start', '')
                            be = brk.get('end', '')
                            if bs and be:
                                break_values.add(f'{bs}-{be}')
                    elif '휴무' in desc:
                        closed_days.append(day)

            except Exception as e:
                print(f'파싱 오류: {e}')

        # 휴무일: comingRegularClosedDays 우선, 없으면 description에서 수집한 날짜
        closed_match = re.search(r'"comingRegularClosedDays"\s*:\s*"([^"]*)"', src)
        if closed_match and closed_match.group(1).strip():
            closed_str = closed_match.group(1).strip()
        elif closed_days:
            closed_str = ', '.join(closed_days)
        else:
            closed_str = '없음'

        if time_values:
            break_str = ', '.join(sorted(break_values)) if break_values else '없음'
            return time_values, closed_str, break_str

        # 방법 B: DOM 텍스트 패턴 (fallback)
        fallback_closed = ''
        for day in ['월','화','수','목','금','토','일']:
            m = re.search(rf'{day}\s+([\d:]+\s*[-~]\s*[\d:]+)', src)
            if m:
                time_values.add(m.group(1).strip().replace(' ',''))
            if re.search(rf'{day}\s+(휴무|정기휴무)', src):
                fallback_closed += f'{day}, '

        result_closed = fallback_closed.rstrip(', ') or closed_str
        return time_values, result_closed if result_closed else '없음', '없음'

    except Exception as e:
        print(f'오류: {e}')
        return set(), '없음', '없음'
    finally:
        driver.switch_to.default_content()

In [ ]:
# ✅ STEP 7: 테스트 (1개만 먼저)
test_name = all_data[1][B_COL-1]
print(f'테스트: {test_name}')
hours = get_hours_from_naver(test_name, driver)
print(f'결과: {hours}')

테스트: 크리미
[크리미] place_id: 1238077488
결과: ({'00:00-24:00'}, '없음', '없음')


In [ ]:
# ✅ STEP 8: 전체 실행
START_ROW  = 601
END_ROW    = 800
DELAY_SEC  = 3
BATCH      = 10
CLOSED_COL = 22   # V열 - 휴무일
HOURS_COL  = 23   # W열 - 영업시간
BREAK_COL  = 24   # X열 - 브레이크타임

updated, skipped, failed = 0, 0, 0
batch_updates = []

all_data = sheet.get_all_values()
total_range = END_ROW - START_ROW + 1

print(f'처리 범위: {START_ROW}행 ~ {END_ROW}행 (총 {total_range}개)')
print('=' * 50)

for i in range(START_ROW, END_ROW + 1):
    row = all_data[i-1] if i <= len(all_data) else []
    place_name    = row[B_COL-1].strip()      if len(row) >= B_COL     else ''
    gu            = row[K_COL-1].strip()      if len(row) >= K_COL     else ''
    current_hours = row[HOURS_COL-1].strip()  if len(row) >= HOURS_COL else ''

    if current_hours:
        print(f'[{i}] ⏭  {place_name} : 스킵 (이미 있음)')
        skipped += 1
        continue

    if not place_name:
        print(f'[{i}] ⚠️  가게명 없음 : 스킵')
        continue

    hours_set, closed_str, break_str = get_hours_from_naver(place_name, driver, gu=gu)

    if hours_set:
        hours_str = ', '.join(sorted(hours_set))
        batch_updates.append((i, hours_str, closed_str, break_str))
        print(f'[{i}] ✅  {place_name}({gu}) : 영업({hours_str}) / 휴무({closed_str}) / 브레이크({break_str})')
        updated += 1
    else:
        print(f'[{i}] ❌  {place_name}({gu}) : 실패')
        failed += 1

    if len(batch_updates) >= BATCH:
        for row_num, hrs, cls, brk in batch_updates:
            sheet.update_cell(row_num, CLOSED_COL, cls)
            sheet.update_cell(row_num, HOURS_COL,  hrs)
            sheet.update_cell(row_num, BREAK_COL,  brk)
        print(f'\n💾 {len(batch_updates)}개 저장 완료\n')
        batch_updates = []

    time.sleep(DELAY_SEC)

if batch_updates:
    for row_num, hrs, cls, brk in batch_updates:
        sheet.update_cell(row_num, CLOSED_COL, cls)
        sheet.update_cell(row_num, HOURS_COL,  hrs)
        sheet.update_cell(row_num, BREAK_COL,  brk)
    print(f'\n💾 {len(batch_updates)}개 저장 완료\n')

print('=' * 50)
print(f'완료.  ✅ 성공: {updated}  ❌ 실패: {failed}  ⏭ 스킵: {skipped}')

처리 범위: 601행 ~ 800행 (총 200개)
[이태리횟집] place_id 찾기 실패
[601] ❌  이태리횟집(부산진구) : 실패
[카레온 경성대 본점] place_id: 1705610773
[602] ✅  카레온 경성대 본점(남구) : 영업(10:00-20:00) / 휴무(토) / 브레이크(없음)
[고반식당 부산장산역점] place_id: 1692122285
[603] ✅  고반식당 부산장산역점(해운대구) : 영업(16:00-01:00) / 휴무(없음) / 브레이크(없음)
[스티켓 직달사천] place_id: 1270518956
[604] ✅  스티켓 직달사천(부산진구) : 영업(11:30-21:00) / 휴무(없음) / 브레이크(없음)
[구포짬뽕 김해공항점] place_id: 2019040015
[605] ✅  구포짬뽕 김해공항점(강서구) : 영업(11:00-20:30) / 휴무(월) / 브레이크(없음)
[쿠시마루] place_id 찾기 실패
[606] ❌  쿠시마루(수영구) : 실패
[블루보틀 부산 민락 카페] place_id: 1865096060
[607] ✅  블루보틀 부산 민락 카페(수영구) : 영업(09:00-22:00) / 휴무(없음) / 브레이크(없음)
[소쿠리 본점] place_id: 1348525737
[608] ✅  소쿠리 본점(연제구) : 영업(00:00-24:00) / 휴무(없음) / 브레이크(없음)
[샤브원 수영구광안점] place_id 찾기 실패
[609] ❌  샤브원 수영구광안점(수영구) : 실패
[해운대 해물칼국수] place_id: 1656626431
[610] ✅  해운대 해물칼국수(기장군) : 영업(11:00-20:00) / 휴무(없음) / 브레이크(없음)
[MABINOGI CAFE for Milletian] place_id 찾기 실패
[611] ❌  MABINOGI CAFE for Milletian(해운대구) : 실패
[코나바리] place_id: 1530235397
[612] ✅  코나바리(수영구) : 영업(18

In [ ]:
place_id = '16720361'

driver.get(f'https://pcmap.place.naver.com/restaurant/{place_id}/home')
time.sleep(5)
src = driver.page_source

raw = extract_json_array(src, 'businessHours')
if raw:
    for item in raw:
        print(item)
        print('---')
else:
    print('businessHours 없음')

print('\n=== comingRegularClosedDays ===')
closed_match = re.search(r'"comingRegularClosedDays"\s*:\s*"([^"]*)"', src)
print(closed_match.group(1) if closed_match else '없음')

print('\n=== freeText ===')
free_match = re.search(r'"freeText"\s*:\s*"([^"]*)"', src)
print(free_match.group(1) if free_match else '없음')

{'__typename': 'WorkingHoursInfo', 'day': '수', 'businessHours': {'__typename': 'StartEndTime', 'start': '09:00', 'end': '15:30'}, 'breakHours': [], 'description': None, 'lastOrderTimes': [{'__typename': 'LastOrderTimes', 'type': '영업시간', 'time': '15:25'}], 'showEndsNextDay': False}
---
{'__typename': 'WorkingHoursInfo', 'day': '목', 'businessHours': {'__typename': 'StartEndTime', 'start': '09:00', 'end': '15:30'}, 'breakHours': [], 'description': None, 'lastOrderTimes': [{'__typename': 'LastOrderTimes', 'type': '영업시간', 'time': '15:25'}], 'showEndsNextDay': False}
---
{'__typename': 'WorkingHoursInfo', 'day': '금', 'businessHours': {'__typename': 'StartEndTime', 'start': '09:00', 'end': '15:30'}, 'breakHours': [], 'description': None, 'lastOrderTimes': [{'__typename': 'LastOrderTimes', 'type': '영업시간', 'time': '15:25'}], 'showEndsNextDay': False}
---
{'__typename': 'WorkingHoursInfo', 'day': '토', 'businessHours': {'__typename': 'StartEndTime', 'start': '09:00', 'end': '15:30'}, 'breakHours'